# Module 3c: Running VCell Simulations with PyVCell

**Scripts covered:** `3c_run_pyvcell/1_run_CPC_model.py`, `3c_run_pyvcell/2_run_fielddata_from_sim_workflow.py`, `3c_run_pyvcell/3_run_CPC_transition_model.py`

## Purpose

**PyVCell** is a Python API that lets you load, configure, and run VCell VCML models programmatically — without the GUI or the HPC workflow. This is the recommended approach for:

- Quick single-simulation runs during development
- Testing model changes before committing to HPC parameter scans
- Prototyping new initial conditions or parameter values
- Generating field data outputs that are used as initial conditions for follow-on simulations

### PyVCell vs. HPC CLI

| | PyVCell (Module 3c) | HPC/SLURM (Module 3b) |
|---|---|---|
| Setup | `pip install pyvcell` | Singularity + SLURM |
| Use case | Development, single runs | Production, parameter scans |
| Scale | Single simulation | Many simulations in parallel |
| Output | Zarr + HDF5 | `reports.h5` |

---

## Setup

In [ ]:
import pyvcell.vcml as vc
import time

# Update this path to your local VCML file
vcml_file = "/path/to/VCell_Analysis/vcell_models/vcml/_09_16_25_CPC_metacentric_relaxed_model_v2.vcml"

## Part 1: Loading a VCML model

A **VCML file** (Virtual Cell Markup Language) is an XML file that encodes:
- The biochemical reaction network (species, reactions, rate laws)
- The spatial geometry (compartment shapes and sizes)
- Simulation configurations (duration, timestep, mesh size, output variables)
- Initial conditions and parameters

`vc.load_vcml_file()` parses the XML and returns a `BioModel` object containing all of this information.

In [ ]:
# Load the biomodel from VCML
bio_model = vc.load_vcml_file(vcml_file)

# The biomodel contains 'applications' — each application has a geometry + simulations
print(f"Number of applications: {len(bio_model.applications)}")
for i, app in enumerate(bio_model.applications):
    print(f"  Application {i}: {app.name}")
    for j, sim in enumerate(app.simulations):
        print(f"    Simulation {j}: {sim.name}")

## Part 2: Inspecting and Configuring Simulation Parameters

Before running, you can inspect and modify simulation settings. The most commonly adjusted parameters are:

- **`duration`**: Total model time to simulate (in seconds). The relaxed model typically runs for 200–500 s to reach steady state.
- **`output_time_step`**: How frequently to save output (in seconds). Smaller = more data but larger files.
- **`mesh_size`**: The spatial grid resolution. The standard metacentric chromosome geometry is 144 × 52.

In [ ]:
# Access the first simulation in the first application
sim = bio_model.applications[0].simulations[0]

print(f"Simulation name: {sim.name}")
print(f"Current mesh size: {sim.mesh_size}")
print(f"Current duration: {sim.duration} s")
print(f"Current output timestep: {sim.output_time_step} s")

In [ ]:
# Modify simulation parameters before running
# For a quick test run, use a short duration:
bio_model.applications[0].simulations[0].duration = 20.0       # seconds of model time
bio_model.applications[0].simulations[0].output_time_step = 10.0  # save every 10 s

# For a full production run, use:
# bio_model.applications[0].simulations[0].duration = 500.0
# bio_model.applications[0].simulations[0].output_time_step = 10.0

## Part 3: Running the Simulation

`vc.simulate()` sends the model to the VCell solver and blocks until the simulation completes. It returns a `SimulationResult` object containing the output data.

**Note:** Solving 2D reaction-diffusion PDEs takes time. For the 144×52 mesh over 500 s, expect 5–30 minutes depending on your machine. The short 20 s test run below should complete in under a minute.

In [ ]:
start_time = time.perf_counter()

# Run the simulation
# sim.name identifies which simulation configuration to use
result = vc.simulate(biomodel=bio_model, simulation=sim.name)

elapsed = time.perf_counter() - start_time
print(f"Simulation completed in {elapsed:.1f} seconds")
print(f"Output saved to: {result.solver_output_dir}")

## Part 4: Inspecting the Results

The `result` object provides access to all output data through `channel_data`. Each channel corresponds to one molecular species.

In [ ]:
# List all available output channels (species)
channels = [c.label for c in result.channel_data]
print(f"Total output channels: {len(channels)}")
print("\nAvailable species:")
for ch in channels:
    print(f"  {ch}")

## Part 5: Visualization

PyVCell provides a built-in `plotter` object with two key methods:

### `plot_slice_2d(time_index, channel_id)`
Shows a 2D heatmap of a species' concentration at a given timepoint. The X axis is the long axis of the chromosome (3.6 µm for the metacentric model), and the Y axis is the short axis (1.3 µm).

In [ ]:
# Plot CPCa (active CPC) concentration at timepoint index 3 (i.e., t = 30 s if dt_out = 10 s)
result.plotter.plot_slice_2d(time_index=3, channel_id="CPCa")

In [ ]:
# Try different species and timepoints
result.plotter.plot_slice_2d(time_index=1, channel_id="pH3")

### `plot_concentrations()`

Plots spatially averaged concentration vs. time for all channels — useful for quickly checking whether the simulation reached steady state.

In [ ]:
result.plotter.plot_concentrations()

## Part 6: Cleanup

PyVCell writes temporary files to disk during the simulation. `result.cleanup()` removes these. Call this when you're done with the result to avoid accumulating large temporary files.

In [ ]:
result.cleanup()

---

## Script 2: Using Field Data from a Previous Simulation as Initial Conditions

**Script:** `3c_run_pyvcell/2_run_fielddata_from_sim_workflow.py`

This is a more advanced workflow that uses the **spatial output** of one simulation as the **initial condition** of a subsequent simulation. This is used for the relaxed → tensed → transition model sequence:

1. Run the relaxed model to steady state (Module 3b or 3c)
2. Take the steady-state spatial distribution of all species
3. Load those distributions as initial conditions in the transition model
4. Run the transition model to simulate the kinetics of CPC redistribution as tension develops

This avoids having to manually specify 2D initial conditions for 40+ species — instead, we start from a biologically realistic starting state (the relaxed steady state).

```python
# Conceptual workflow from 2_run_fielddata_from_sim_workflow.py

# Step 1: Load the field data output from a previous simulation
# (This is the spatially resolved concentration at the last timepoint)
field_data = vc.load_field_data(previous_sim_output_dir)

# Step 2: Load the transition model
transition_model = vc.load_vcml_file(transition_vcml_file)

# Step 3: Set the field data as initial conditions
transition_model.applications[0].set_field_data_initial_conditions(field_data)

# Step 4: Run
result = vc.simulate(biomodel=transition_model, simulation=sim.name)
```

---

## Script 3: Running the Transition Model

**Script:** `3c_run_pyvcell/3_run_CPC_transition_model.py`

The transition model (`_09_16_25_CPC_metacentric_tensed_model_v2.vcml` or the transition variant) adds kinetochore-pulling dynamics that simulate chromosome biorientation. It is run after the relaxed model reaches steady state.

Key differences from the relaxed model:
- Kinetochore positions shift (the KK distance increases: 0.575 µm → 1.15 µm)
- HASPIN activity is modulated by tension-dependent phosphorylation
- The geometry may be updated to reflect the tensed chromosome shape

Usage is identical to Script 1 — just point `vcml_file` to the tensed or transition VCML.

---

## Converting PyVCell output for downstream analysis

PyVCell output is stored in a Zarr/HDF5 format that can be converted to the CSV format expected by Module 4 using `hdf5_converter.py`. See the Module 4 tutorial for details.